# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available RecordSets
print("\nAvailable RecordSets (referenced by @id):")
record_set_ids = []
for record_set in dataset.metadata.record_sets:
    print(f"  - {record_set['@id']}: {record_set.get('name', 'No name')}\n    Description: {record_set.get('description', 'No description')}")
    record_set_ids.append(record_set['@id'])

# For each RecordSet, list its fields and columns by @id
for record_set in dataset.metadata.record_sets:
    print(f"\nRecordSet {record_set['@id']} fields and columns:")
    print("Fields:")
    for field in record_set.get('fields', []):
        print(f"  - {field['@id']}: {field.get('name', '')}")
    print("Columns:")
    for col in record_set.get('columns', []):
        print(f"  - {col['@id']}: {col.get('name', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use record set and field/column `@id`s from the overview.

In [ ]:
# Define your wanted record sets (by @id) for extraction
record_sets = record_set_ids  # gathered above
dataframes = {}

for record_set_id in record_sets:
    try:
        records_list = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records_list)
        dataframes[record_set_id] = df
        print(f"\nLoaded records for {record_set_id}, fields: {df.columns.tolist()}")
        display(df.head())
    except Exception as e:
        print(f"\nNo records found or error loading for {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration, pick the first (presumably main) record set as the main table
if len(record_set_ids) > 0:
    main_rs = record_set_ids[0]
    df = dataframes[main_rs]
    print(f"Columns in '{main_rs}': {df.columns.tolist()}")
    # Attempt to pick a likely numeric column
    import numpy as np
    candidate_numeric_fields = [c for c in df.columns if df[c].dtype in [np.int64, np.float64] or pd.api.types.is_numeric_dtype(df[c])]
    if not candidate_numeric_fields and len(df) > 0:
        # Try to infer numeric columns if all read as object
        for c in df.columns:
            try:
                df[c] = pd.to_numeric(df[c])
                candidate_numeric_fields.append(c)
            except:
                continue
    if candidate_numeric_fields:
        numeric_field = candidate_numeric_fields[0]
        threshold = df[numeric_field].mean()  # use mean as threshold
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records in '{main_rs}' where {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Try grouping by a possibly categorical column
        group_field_candidates = [c for c in df.columns if c != numeric_field and df[c].dtype == object]
        for group_field in group_field_candidates:
            if df[group_field].nunique() < 20 and df[group_field].nunique() > 1:
                try:
                    grouped_df = filtered_df.groupby(group_field).mean()
                    print(f"Grouped data by {group_field}:")
                    display(grouped_df.head())
                    break
                except Exception:
                    continue
    else:
        print("No numeric fields found in main record set. Skipping EDA.")
else:
    print("No record sets found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization for the main numeric field (if available)
if len(record_set_ids) > 0 and candidate_numeric_fields:
    fig, ax = plt.subplots(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, ax=ax)
    ax.set_title(f"Distribution of {numeric_field} in RecordSet {main_rs}")
    plt.show()

    # If there is a group field for colored boxplot
    if group_field_candidates:
        fig, ax = plt.subplots(figsize=(10,6))
        sns.boxplot(x=group_field_candidates[0], y=numeric_field, data=df, ax=ax)
        ax.set_title(f"{numeric_field} by {group_field_candidates[0]}")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* In this notebook, we loaded the FAIR² Croissant-formatted dataset describing mass spectrometry analysis from human extracellular vesicles.
* We reviewed dataset metadata, explored the available record sets, and extracted tables for further use.
* Sample exploratory data analysis and visualizations illustrated how to filter, normalize, and summarize protein attributes.

Further exploration could include deeper biological or biomarker pattern mining, domain-specific feature engineering, or comparisons across different sample classes.